# Analytical Jupyter Notebook for Telescope-Received Space Data

This notebook turns the earlier **JWST / MAST FITS rendering workflow** into an **analysis-first workflow** for telescope data.

It is designed for:
- **local FITS files** you already have
- **optional MAST download mode** for public astronomy data
- **JWST first**, but the same workflow can often work for HST and other FITS-based image products

## What this notebook does

1. Finds local FITS files or downloads public products from MAST  
2. Inspects FITS structure and selects usable science HDUs  
3. Cleans bad values, estimates background, and removes likely hot pixels  
4. Aligns multiple filters to a common WCS when possible  
5. Produces analysis tables for image statistics and noise  
6. Detects a bright source and measures simple aperture flux  
7. Builds radial profiles around a detected source  
8. Handles FITS cubes or frame stacks for basic time-series analysis  
9. Produces quicklook visualizations and optional RGB composites  
10. Exports summary CSV files for later reuse

## Assumption

This notebook assumes **"JSW" was intended as "JWST"**, based on your reference workflow.


In [ ]:
# Run once in a fresh environment
%pip -q install astroquery astropy reproject photutils scipy scikit-image pillow matplotlib pandas numpy

## Imports

In [ ]:
from pathlib import Path
import warnings
import re
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

from astropy.io import fits
from astropy.stats import sigma_clipped_stats
from astropy.visualization import simple_norm
from astropy.wcs import WCS

from astroquery.mast import Observations
from reproject import reproject_interp
from scipy.ndimage import median_filter

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

plt.rcParams["figure.figsize"] = (8, 8)
plt.rcParams["image.origin"] = "lower"
plt.rcParams["axes.grid"] = False

## Configuration

### Two ways to use this notebook

**Mode A — local FITS files**
- Put FITS files into a folder
- Set `DATA_MODE = "local"`

**Mode B — public data from MAST**
- Search by target name
- Set `DATA_MODE = "mast"`

You can keep the defaults first, then change only what you need.


In [ ]:
DATA_MODE = "local"   # "local" or "mast"

LOCAL_DATA_DIR = Path("./fits_input")
EXPLICIT_FITS_FILES = []  # Example: ["./fits_input/file1.fits"]

MAST_TARGET = "M16"
MAST_RADIUS_DEG = 0.02
MAST_COLLECTION = "JWST"
MAST_MAX_OBS = 10
DOWNLOAD_DIR = Path("./mast_downloads")

PREFERRED_PRODUCT_KEYWORDS = ["calints", "rateints", "cal", "rate", "i2d", "drz", "drc", "sci"]

OUTPUT_DIR = Path("./analysis_output")
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

BACKGROUND_SIGMA = 3.0
BACKGROUND_MAXITERS = 10
HOT_PIXEL_SIZE = 3
HOT_PIXEL_THRESHOLD_SIGMA = 8.0

DETECTION_SIGMA = 5.0
APERTURE_RADIUS = 8.0
RADIAL_BIN_SIZE = 1.0

STRETCH = "asinh"   # "asinh", "sqrt", "log", "linear"
DISPLAY_PERCENTILE = 99.5

MANUAL_FILE_OVERRIDE = {}
MANUAL_CHANNEL_HINTS = {}

## Helper functions

In [ ]:

def list_fits_files(folder: Path):
    if not folder.exists():
        return []
    files = []
    for pat in ("*.fits", "*.fit", "*.fts", "*.fits.gz"):
        files.extend(folder.rglob(pat))
    return sorted(set(files))


def choose_science_hdu(hdul):
    candidates = []
    for idx, hdu in enumerate(hdul):
        data = getattr(hdu, "data", None)
        if data is None or not isinstance(data, np.ndarray) or data.ndim < 2:
            continue
        name = (getattr(hdu, "name", "") or "").upper()
        header = hdu.header
        score = 0
        if name == "SCI":
            score += 100
        if idx == 0:
            score += 10
        if data.ndim == 2:
            score += 20
        if "WCSAXES" in header or "CTYPE1" in header:
            score += 30
        if data.shape[-2] > 128 and data.shape[-1] > 128:
            score += 20
        candidates.append((score, idx, hdu))
    if not candidates:
        return None, None, None
    candidates.sort(reverse=True, key=lambda x: x[0])
    _, idx, hdu = candidates[0]
    return idx, hdu.data, hdu.header


def robust_clean(data):
    x = np.asarray(data, dtype=np.float32).copy()
    bad = ~np.isfinite(x)
    if bad.any():
        good = ~bad
        fill = np.nanmedian(x[good]) if good.any() else 0.0
        x[bad] = fill
    return x


def to_frame_cube(arr: np.ndarray) -> np.ndarray:
    a = np.asarray(arr)
    a = np.squeeze(a)

    if a.ndim == 2:
        return a[None, :, :]

    if a.ndim == 3:
        # Usually (n, y, x) or (y, x, n)
        if a.shape[0] <= 10000 and a.shape[1] > 8 and a.shape[2] > 8:
            return a.astype(np.float32)
        if a.shape[-1] <= 10000 and a.shape[0] > 8 and a.shape[1] > 8:
            return np.moveaxis(a, -1, 0).astype(np.float32)
        raise ValueError(f"Unclear 3D layout: {a.shape}")

    if a.ndim == 4:
        # Common JWST-like layout: (nints, ngroups, y, x)
        return np.nanmean(a, axis=1).astype(np.float32)

    raise ValueError(f"Unsupported data shape for cube analysis: {a.shape}")


def background_subtract(data, sigma=BACKGROUND_SIGMA, maxiters=BACKGROUND_MAXITERS):
    mean, median, std = sigma_clipped_stats(data, sigma=sigma, maxiters=maxiters)
    return data - median, {"mean": float(mean), "median": float(median), "std": float(std)}


def remove_hot_pixels(data, size=HOT_PIXEL_SIZE, threshold_sigma=HOT_PIXEL_THRESHOLD_SIGMA):
    med = median_filter(data, size=size)
    resid = data - med
    _, _, std = sigma_clipped_stats(resid, sigma=3.0, maxiters=5)
    mask = resid > threshold_sigma * std
    out = data.copy()
    out[mask] = med[mask]
    return out


def robust_limits(arr, lo=2.0, hi=99.7):
    vmin, vmax = np.nanpercentile(arr, [lo, hi])
    if not np.isfinite(vmin) or not np.isfinite(vmax) or vmax <= vmin:
        vmin, vmax = np.nanmin(arr), np.nanmax(arr)
    return float(vmin), float(vmax)


def normalize_channel(data, low=1.0, high=99.7, stretch="asinh"):
    x = np.array(data, dtype=np.float32)
    lo = np.nanpercentile(x, low)
    hi = np.nanpercentile(x, high)
    if hi <= lo:
        hi = lo + 1e-6
    x = np.clip((x - lo) / (hi - lo), 0, 1)
    if stretch == "sqrt":
        x = np.sqrt(x)
    elif stretch == "log":
        x = np.log1p(1000 * x) / np.log1p(1000)
    elif stretch == "asinh":
        a = 8.0
        x = np.arcsinh(a * x) / np.arcsinh(a)
    elif stretch != "linear":
        raise ValueError(f"Unknown stretch: {stretch}")
    return np.clip(x, 0, 1)


def show_frame(data, title=None, cmap="gray", percentile=DISPLAY_PERCENTILE, figsize=(7, 7)):
    vmin, vmax = robust_limits(data, 100 - percentile, percentile)
    plt.figure(figsize=figsize)
    plt.imshow(data, cmap=cmap, vmin=vmin, vmax=vmax)
    plt.colorbar(label="signal")
    if title:
        plt.title(title)
    plt.tight_layout()
    plt.show()


def detect_filter_name(header, path=None):
    keys = ["FILTER", "PUPIL", "FILTNAM1", "FILTNAM2", "FILTER1", "FILTER2"]
    vals = []
    for k in keys:
        if k in header and str(header[k]).strip():
            vals.append(str(header[k]).strip().lower())
    if path is not None:
        stem = Path(path).stem.lower()
        vals.extend(re.findall(r"f\d{3}[wmn]", stem))
    vals = [v for v in vals if v not in {"clear", "none", "nan"}]
    if vals:
        for v in vals:
            if re.match(r"f\d{3}[wmn]", v):
                return v
        return vals[0]
    return Path(path).stem.lower() if path is not None else "unknown"


def wavelength_key(name):
    m = re.search(r"f(\d{3})([wmn])", str(name).lower())
    return int(m.group(1)) if m else 9999


def auto_assign_rgb(filter_names):
    ordered = sorted(filter_names, key=wavelength_key)
    n = len(ordered)
    if n == 1:
        return {"R": ordered[0], "G": ordered[0], "B": ordered[0]}
    if n == 2:
        return {"R": ordered[1], "G": ordered[0], "B": ordered[0]}
    if n == 3:
        return {"B": ordered[0], "G": ordered[1], "R": ordered[2]}
    return {"B": ordered[0], "G": ordered[n // 2], "R": ordered[-1]}


def find_local_files():
    if EXPLICIT_FITS_FILES:
        files = [Path(x) for x in EXPLICIT_FITS_FILES]
    else:
        files = list_fits_files(LOCAL_DATA_DIR)
    return [f for f in files if f.exists()]


def search_and_download_mast(target=MAST_TARGET, radius_deg=MAST_RADIUS_DEG,
                             collection=MAST_COLLECTION, max_obs=MAST_MAX_OBS,
                             download_dir=DOWNLOAD_DIR):
    download_dir.mkdir(exist_ok=True, parents=True)
    print(f"Searching MAST for target={target!r}, collection={collection!r} ...")
    obs = Observations.query_object(target, radius=f"{radius_deg} deg")
    if len(obs) == 0:
        raise RuntimeError("No observations found.")

    df = obs.to_pandas()
    if "obs_collection" in df.columns:
        df = df[df["obs_collection"].astype(str).str.upper() == str(collection).upper()]
    if len(df) == 0:
        raise RuntimeError(f"No observations left after filtering to collection={collection!r}")

    df = df.head(max_obs)
    print(f"Found {len(df)} matching observations")

    products = Observations.get_product_list(obs[np.isin(obs["obsid"], df["obsid"].tolist())])
    pdf = products.to_pandas()

    keep = pd.Series(False, index=pdf.index)
    for col in ("productFilename", "description", "productType", "dataproduct_type"):
        if col in pdf.columns:
            s = pdf[col].astype(str).str.lower()
            for kw in PREFERRED_PRODUCT_KEYWORDS:
                keep = keep | s.str.contains(kw, na=False)

    if "productFilename" in pdf.columns:
        keep = keep | pdf["productFilename"].astype(str).str.lower().str.endswith(".fits")

    pdf = pdf[keep].copy()
    if len(pdf) == 0:
        raise RuntimeError("No suitable FITS-like products found after heuristic filtering.")

    print(f"Downloading {len(pdf)} products ...")
    downloaded = Observations.download_products(
        products[np.isin(products["obsID"], pdf["obsID"].tolist())],
        download_dir=str(download_dir),
        cache=True,
        mrp_only=False
    )

    ddf = downloaded.to_pandas()
    local_paths = []
    for col in ("Local Path", "Local_Path", "local_path"):
        if col in ddf.columns:
            local_paths.extend([Path(p) for p in ddf[col].dropna().tolist()])

    out = []
    for p in local_paths:
        sp = str(p).lower()
        if p.exists() and (sp.endswith(".fits") or sp.endswith(".fits.gz") or p.suffix.lower() in {".fits", ".gz"}):
            out.append(p)
    return sorted(set(out))


def image_statistics(frame):
    mean, median, std = sigma_clipped_stats(frame, sigma=3.0, maxiters=5)
    q01, q05, q50, q95, q99 = np.nanpercentile(frame, [1, 5, 50, 95, 99])
    return {
        "shape_y": int(frame.shape[-2]),
        "shape_x": int(frame.shape[-1]),
        "mean": float(mean),
        "median": float(median),
        "std": float(std),
        "min": float(np.nanmin(frame)),
        "max": float(np.nanmax(frame)),
        "p01": float(q01),
        "p05": float(q05),
        "p50": float(q50),
        "p95": float(q95),
        "p99": float(q99),
    }


def centroid_bright_source(frame):
    x = np.nan_to_num(frame, nan=np.nanmedian(frame))
    x = x - np.nanmin(x)
    y_idx, x_idx = np.indices(x.shape)
    total = x.sum()
    if total <= 0:
        return frame.shape[1] / 2, frame.shape[0] / 2
    x0 = (x_idx * x).sum() / total
    y0 = (y_idx * x).sum() / total
    return float(x0), float(y0)


def circular_aperture_flux(frame, x0, y0, r):
    y, x = np.indices(frame.shape)
    mask = (x - x0) ** 2 + (y - y0) ** 2 <= r ** 2
    return float(np.nansum(frame[mask])), mask


def estimate_local_background(frame, x0, y0, r_in, r_out):
    y, x = np.indices(frame.shape)
    rr = (x - x0) ** 2 + (y - y0) ** 2
    ann = (rr >= r_in ** 2) & (rr <= r_out ** 2)
    vals = frame[ann]
    vals = vals[np.isfinite(vals)]
    if len(vals) == 0:
        return 0.0
    return float(np.nanmedian(vals))


def aperture_photometry_with_background(frame, x0, y0, r=APERTURE_RADIUS, r_in=None, r_out=None):
    if r_in is None:
        r_in = 1.5 * r
    if r_out is None:
        r_out = 2.5 * r
    raw_flux, mask = circular_aperture_flux(frame, x0, y0, r)
    bg = estimate_local_background(frame, x0, y0, r_in, r_out)
    area = int(mask.sum())
    net_flux = raw_flux - bg * area
    return {
        "x0": float(x0),
        "y0": float(y0),
        "radius": float(r),
        "raw_flux": float(raw_flux),
        "background_per_pixel": float(bg),
        "aperture_area": int(area),
        "net_flux": float(net_flux),
    }


def radial_profile(frame, x0, y0, binsize=RADIAL_BIN_SIZE, max_r=None):
    y, x = np.indices(frame.shape)
    r = np.sqrt((x - x0) ** 2 + (y - y0) ** 2)
    if max_r is None:
        max_r = np.nanmax(r)
    edges = np.arange(0, max_r + binsize, binsize)
    centers = 0.5 * (edges[:-1] + edges[1:])
    values = []
    counts = []
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (r >= lo) & (r < hi)
        vals = frame[m]
        vals = vals[np.isfinite(vals)]
        counts.append(int(vals.size))
        values.append(float(np.nanmean(vals)) if vals.size else np.nan)
    return pd.DataFrame({"radius": centers, "mean_signal": values, "n_pix": counts})


def load_record_image(rec):
    with fits.open(rec["path"]) as hdul:
        data = robust_clean(hdul[rec["hdu_index"]].data)
    return data


def build_record_catalog(fits_files):
    records = []
    for path in fits_files:
        try:
            with fits.open(path) as hdul:
                idx, data, header = choose_science_hdu(hdul)
                if data is None:
                    print(f"Skipping {path.name}: no usable 2D image HDU found")
                    continue
                filt = detect_filter_name(header, path)
                obj = str(header.get("OBJECT", ""))
                inst = str(header.get("INSTRUME", ""))
                tel = str(header.get("TELESCOP", ""))
                records.append({
                    "path": str(path),
                    "file_name": path.name,
                    "hdu_index": idx,
                    "filter_name": filt,
                    "shape": tuple(data.shape),
                    "n_dim": int(np.ndim(data)),
                    "object": obj,
                    "instrument": inst,
                    "telescope": tel,
                    "header": header,
                })
        except Exception as e:
            print(f"Skipping {path.name}: {e}")
    catalog = pd.DataFrame(records)
    if len(catalog):
        catalog["n_pix"] = catalog["shape"].apply(lambda s: int(np.prod(s[-2:])))
    return catalog


def align_to_reference(selection_df):
    selected_filters = sorted(selection_df["filter_name"].tolist(), key=wavelength_key)
    if not selected_filters:
        raise RuntimeError("No filters selected for alignment.")
    reference_filter = selected_filters[0]

    loaded = {}
    for _, row in selection_df.iterrows():
        filt = row["filter_name"]
        path = row["path"]
        hdu_index = int(row["hdu_index"])
        with fits.open(path) as hdul:
            data = robust_clean(hdul[hdu_index].data)
            header = hdul[hdu_index].header
        data, stats = background_subtract(data)
        data = remove_hot_pixels(data)
        loaded[filt] = {
            "data": data,
            "header": header,
            "wcs": WCS(header),
            "stats": stats,
            "path": path,
        }

    ref = loaded[reference_filter]
    ref_header = ref["header"]
    ref_shape = ref["data"].shape

    aligned = {}
    for filt, item in loaded.items():
        if filt == reference_filter:
            aligned[filt] = item["data"]
            continue
        try:
            reprojected, footprint = reproject_interp(
                (item["data"], item["wcs"]),
                ref_header,
                shape_out=ref_shape,
            )
            aligned[filt] = robust_clean(reprojected)
            print(f"Aligned {filt} -> {reference_filter}")
        except Exception as e:
            print(f"Could not align {filt}: {e}")

    return reference_filter, loaded, aligned


def make_rgb_quicklook(aligned_data, rgb_assignment, stretch=STRETCH):
    r = normalize_channel(aligned_data[rgb_assignment["R"]], stretch=stretch)
    g = normalize_channel(aligned_data[rgb_assignment["G"]], stretch=stretch)
    b = normalize_channel(aligned_data[rgb_assignment["B"]], stretch=stretch)
    return np.dstack([r, g, b])


def save_dataframe(df, name):
    path = OUTPUT_DIR / name
    df.to_csv(path, index=False)
    print("Saved:", path.resolve())
    return path


## Step 1 — Find FITS data

In [ ]:
if DATA_MODE == "local":
    fits_files = find_local_files()
else:
    fits_files = search_and_download_mast()

print(f"Total FITS files found: {len(fits_files)}")
for f in fits_files[:20]:
    print(" -", f)

if len(fits_files) == 0:
    raise RuntimeError("No FITS files found. Put files into LOCAL_DATA_DIR or switch DATA_MODE to 'mast'.")

## Step 2 — Inspect files and build a catalog

In [ ]:
catalog = build_record_catalog(fits_files)
catalog[["file_name", "filter_name", "hdu_index", "shape", "object", "instrument", "telescope", "n_pix"]]

In [ ]:
if len(catalog) == 0:
    raise RuntimeError("No usable science images were found.")

## Step 3 — Preview one image per filter

In [ ]:
unique_filters = catalog["filter_name"].dropna().unique().tolist()

preview_rows = []
for filt in unique_filters:
    rec = catalog[catalog["filter_name"] == filt].iloc[0]
    data = load_record_image(rec)
    data, stats = background_subtract(data)
    data = remove_hot_pixels(data)
    preview_rows.append((filt, data, stats))

n = len(preview_rows)
fig, axes = plt.subplots(1, n, figsize=(5 * max(n, 1), 5))
if n == 1:
    axes = [axes]

for ax, (filt, data, stats) in zip(axes, preview_rows):
    norm = simple_norm(data, stretch=STRETCH, percent=99.5)
    ax.imshow(data, norm=norm, cmap="gray")
    ax.set_title(f"{filt}\nmedian={stats['median']:.3g}")
    ax.axis("off")

plt.tight_layout()
plt.show()

## Step 4 — Pick one representative file per filter

In [ ]:
best_per_filter = (
    catalog.sort_values("n_pix", ascending=False)
           .drop_duplicates("filter_name")
           .reset_index(drop=True)
)

best_per_filter[["filter_name", "path", "hdu_index", "shape", "n_pix"]]

In [ ]:
selection_rows = []
for _, row in best_per_filter.iterrows():
    filt = row["filter_name"]
    path = MANUAL_FILE_OVERRIDE.get(filt, row["path"])
    selection_rows.append({
        "filter_name": filt,
        "path": path,
        "hdu_index": int(row["hdu_index"]),
    })

selection_df = pd.DataFrame(selection_rows)
selection_df

## Step 5 — Load, clean, and align all filters onto a common WCS

In [ ]:
reference_filter, loaded, aligned = align_to_reference(selection_df)
print("Reference filter:", reference_filter)
print("Aligned filters:", list(aligned.keys()))

## Step 6 — Check aligned previews

In [ ]:
n = len(aligned)
fig, axes = plt.subplots(1, n, figsize=(5 * max(n, 1), 5))
if n == 1:
    axes = [axes]

for ax, filt in zip(axes, aligned.keys()):
    data = aligned[filt]
    norm = simple_norm(data, stretch=STRETCH, percent=99.5)
    ax.imshow(data, norm=norm, cmap="gray")
    ax.set_title(filt)
    ax.axis("off")

plt.tight_layout()
plt.show()

## Step 7 — Build an image statistics table

In [ ]:
stats_rows = []
for filt, frame in aligned.items():
    row = {"filter_name": filt}
    row.update(image_statistics(frame))
    stats_rows.append(row)

stats_df = pd.DataFrame(stats_rows).sort_values("filter_name")
stats_df

In [ ]:
save_dataframe(stats_df, "image_statistics.csv")

## Step 8 — Detect a bright source in the reference filter

This uses a simple flux-weighted centroid.  
For quick engineering analysis, this is often enough to get a usable source position.


In [ ]:
ref_frame = aligned[reference_filter]
x0, y0 = centroid_bright_source(ref_frame)
print(f"Detected centroid in {reference_filter}: x={x0:.2f}, y={y0:.2f}")

plt.figure(figsize=(8, 8))
vmin, vmax = robust_limits(ref_frame, 2, 99.7)
plt.imshow(ref_frame, cmap="gray", vmin=vmin, vmax=vmax)
plt.scatter([x0], [y0], s=80, marker="x")
plt.title(f"Reference filter with detected source: {reference_filter}")
plt.colorbar(label="signal")
plt.show()

## Step 9 — Aperture photometry across filters

In [ ]:
phot_rows = []
for filt, frame in aligned.items():
    phot = aperture_photometry_with_background(frame, x0, y0, r=APERTURE_RADIUS)
    phot["filter_name"] = filt
    phot_rows.append(phot)

phot_df = pd.DataFrame(phot_rows).sort_values("filter_name")
phot_df

In [ ]:
save_dataframe(phot_df, "aperture_photometry.csv")

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(phot_df["filter_name"], phot_df["net_flux"], marker="o")
plt.ylabel("Net aperture flux")
plt.xlabel("Filter")
plt.title("Aperture flux by filter")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Step 10 — Radial profile around the detected source

In [ ]:
profile_df = radial_profile(ref_frame, x0, y0, binsize=RADIAL_BIN_SIZE, max_r=120)
profile_df.head()

In [ ]:
save_dataframe(profile_df, "radial_profile_reference_filter.csv")

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(profile_df["radius"], profile_df["mean_signal"])
plt.xlabel("Radius (pixels)")
plt.ylabel("Mean signal")
plt.title(f"Radial profile — {reference_filter}")
plt.tight_layout()
plt.show()

## Step 11 — Pixel-value histogram and detector-style quick diagnostics

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

axes[0].hist(ref_frame[np.isfinite(ref_frame)].ravel(), bins=200)
axes[0].set_title("Pixel histogram")
axes[0].set_xlabel("Signal")
axes[0].set_ylabel("Count")
axes[0].set_yscale("log")

axes[1].plot(np.nanmedian(ref_frame, axis=0))
axes[1].set_title("Column median trend")
axes[1].set_xlabel("Column")
axes[1].set_ylabel("Median signal")

axes[2].plot(np.nanmedian(ref_frame, axis=1))
axes[2].set_title("Row median trend")
axes[2].set_xlabel("Row")
axes[2].set_ylabel("Median signal")

plt.tight_layout()
plt.show()

## Step 12 — Optional quicklook RGB composite

This is not the main goal of the notebook, but it is useful for sanity-checking registration and filter balance.


In [ ]:
auto_rgb = auto_assign_rgb(list(aligned.keys()))
rgb_assignment = auto_rgb.copy()

for key, ch in MANUAL_CHANNEL_HINTS.items():
    key = key.lower()
    ch = ch.upper()
    if key in aligned and ch in {"R", "G", "B"}:
        rgb_assignment[ch] = key

print("RGB assignment:", rgb_assignment)

rgb_float = make_rgb_quicklook(aligned, rgb_assignment, stretch=STRETCH)

plt.figure(figsize=(10, 10))
plt.imshow(rgb_float)
plt.title("RGB quicklook")
plt.axis("off")
plt.show()

## Step 13 — Optional cube / time-series analysis

If the selected file contains a frame stack or data cube, this section extracts a simple light curve and centroid drift estimate.


In [ ]:
cube_candidate_path = selection_df.iloc[0]["path"]
cube_candidate_hdu = int(selection_df.iloc[0]["hdu_index"])

with fits.open(cube_candidate_path) as hdul:
    raw_cube_data = robust_clean(hdul[cube_candidate_hdu].data)

cube = to_frame_cube(raw_cube_data)
cube.shape

In [ ]:
if cube.shape[0] <= 1:
    print("This file behaves like a single image, so no time-series is available.")
else:
    frame_rows = []
    for i, frame in enumerate(cube):
        clean = robust_clean(frame)
        clean, bg_stats = background_subtract(clean)
        clean = remove_hot_pixels(clean)

        fx0, fy0 = centroid_bright_source(clean)
        phot = aperture_photometry_with_background(clean, fx0, fy0, r=APERTURE_RADIUS)

        frame_rows.append({
            "frame_index": i,
            "centroid_x": fx0,
            "centroid_y": fy0,
            "median": bg_stats["median"],
            "std": bg_stats["std"],
            "raw_flux": phot["raw_flux"],
            "net_flux": phot["net_flux"],
        })

    cube_df = pd.DataFrame(frame_rows)
    display(cube_df.head())

    save_dataframe(cube_df, "cube_time_series.csv")

    fig, axes = plt.subplots(3, 1, figsize=(10, 10), sharex=True)

    axes[0].plot(cube_df["frame_index"], cube_df["net_flux"], marker="o")
    axes[0].set_ylabel("Net flux")
    axes[0].set_title("Light curve")

    axes[1].plot(cube_df["frame_index"], cube_df["centroid_x"], label="x")
    axes[1].plot(cube_df["frame_index"], cube_df["centroid_y"], label="y")
    axes[1].set_ylabel("Centroid")
    axes[1].legend()

    axes[2].plot(cube_df["frame_index"], cube_df["std"], label="background std")
    axes[2].set_xlabel("Frame index")
    axes[2].set_ylabel("Noise")
    axes[2].legend()

    plt.tight_layout()
    plt.show()

## Step 14 — Save a run summary

In [ ]:
run_summary = {
    "data_mode": DATA_MODE,
    "n_input_files": int(len(fits_files)),
    "n_catalog_rows": int(len(catalog)),
    "reference_filter": reference_filter,
    "aligned_filters": list(aligned.keys()),
    "aperture_radius": APERTURE_RADIUS,
    "output_dir": str(OUTPUT_DIR.resolve()),
}

summary_path = OUTPUT_DIR / "run_summary.json"
summary_path.write_text(json.dumps(run_summary, indent=2), encoding="utf-8")
print(summary_path.resolve())
print(json.dumps(run_summary, indent=2))

## Notes and next extensions

### When this notebook is most useful
- telescope image quality checks
- bright-source stability checks
- quick multi-filter comparison
- simple engineering diagnostics on FITS image products
- first-pass cube analysis

### Good next upgrades
- source detection with `photutils`
- PSF fitting
- WCS overlays and sky coordinates
- background models beyond a single median
- more formal light-curve extraction
- FFT or striping diagnostics for detector artifacts
- batch processing across many FITS products
